# 📅 smartbuy Seasonality — How the Web App Was Built

This notebook documents the **production of the Streamlit web app** started with `streamlit run app.py`.
It walks through every stage: from the 100 raw CSV files, through the data pipeline and the KPI
definitions, to the multipage layout, the visual design system and the interactive features.

> The code cells reproduce the app's data logic with plain **pandas / numpy / plotly**, so they run
> without Streamlit. Streamlit-specific code is shown as Markdown snippets taken from the app files.

**Contents**
1. Business problem & goal
2. Project structure
3. Data preparation (combine → reshape → fill gaps)
4. KPI design
5. Priority groups & the long table for charts
6. Visual design system
7. App architecture: entry point, navigation & sidebar filters
8. The four pages
9. Performance & robustness
10. How to run it & lessons learned

## 1. Business problem & goal

### 1.1 The problem
smartbuy's **Content Department** identified seasonal product categories *manually*, by eyeballing
charts. That is slow, subjective and impossible to repeat consistently.

### 1.2 The goal of the app
Give business users a self-service tool that answers, with **measurable and repeatable KPIs**:

- *Which* of the 100 categories are seasonal?
- *When* do they peak (month and season)?
- *Which* ones deserve campaign planning first?

### 1.3 Data available
- 100 CSV files (`data/pis_1.csv` … `data/pis_100.csv`), one per product category.
- Columns: `category`, `month`, `year`, `clicks` — monthly clicks for **2021**.

## 2. Project structure

The app is split into small files with one responsibility each:

```
smartbuy-seasonality-dashboard/
├── app.py                  # entry point: page config, navigation, shared sidebar filters
├── ui.py                   # design system: CSS, colors, hero/cards, chart styling, sidebar filters
├── utils.py                # data loading and all KPI calculations (single source of truth)
├── views/
│   ├── home.py             # project summary, headline numbers, links to the other pages
│   ├── kpis.py             # KPI cards, busiest months, priority matrix, priority list
│   ├── top_clicks.py       # top-n categories by clicks (slider)
│   └── seasonality.py      # season patterns, most seasonal, category deep dive, similar categories
├── app_v1/                 # first version of the app (kept for reference)
├── .streamlit/config.toml  # theme (colors) and toolbar settings
├── data/
│   ├── pis_1.csv … pis_100.csv
│   └── combined_categories.csv   # the 100 files stacked into one (1,200 rows)
└── notebooks/              # exploration, analysis and this notebook
```

**Design decision:** all calculations live in `utils.py` and all styling lives in `ui.py`. The pages
only *display* results, so a KPI is defined once and every page shows the same number.

## 3. Data preparation

### 3.1 Combine the 100 files
The first step (done in the exploration notebook `smartbuy_seasonality_day1.ipynb`) was to stack
the 100 files into one table and save it, so the app only has to read **one** file at start-up.

In [ ]:
# Imports used throughout this notebook
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

DATA = Path("../data")   # the notebook lives in notebooks/, the data one level up

In [ ]:
# Read every category file and stack them below each other
dfs = [pd.read_csv(DATA / f"pis_{i}.csv") for i in range(1, 101)]
df = pd.concat(dfs, ignore_index=True)

# Sanity checks: 100 categories x 12 months = 1,200 rows
print("rows:", len(df))
print("categories:", df["category"].nunique())
print("rows per category:", df.groupby("category").size().unique())
print("missing clicks:", df["clicks"].isna().sum())
df.head()

The combined file is saved once and reused by the app:

```python
df.to_csv("data/combined_categories.csv", index=False)
```

### 3.2 Reshape to a wide table
For seasonality we need **one row per category and one column per month**, so the table is pivoted.

In [ ]:
df = pd.read_csv(DATA / "combined_categories.csv")   # what the app actually loads

monthly_raw = df.pivot(index="category", columns="month", values="clicks")
print("missing values per month:")
monthly_raw.isna().sum().to_frame("missing").T

### 3.3 Fill missing months — *circular* interpolation
A few months have no clicks recorded. Dropping them would break the 12-month profile, so they are
**estimated from their neighbouring months**.

The twist: the year is a **circle** — December and January are neighbours. A normal
`interpolate()` cannot fill a gap in January (no value before it). The trick used in the app:

1. Repeat the 12 months three times (36 values),
2. interpolate linearly,
3. keep only the middle copy.

In [ ]:
def fill_circular(row):
    # Fill missing months; January and December are treated as neighbours
    tripled = pd.concat([row, row, row], ignore_index=True)
    filled = tripled.interpolate()
    return pd.Series(filled.iloc[12:24].values, index=row.index)

monthly = monthly_raw.apply(fill_circular, axis=1)

print("missing before:", int(monthly_raw.isna().sum().sum()),
      "| missing after:", int(monthly.isna().sum().sum()))

# Show the categories that needed estimates
filled_rows = monthly_raw.index[monthly_raw.isna().any(axis=1)]
monthly_raw.loc[filled_rows].head()

## 4. KPI design

The core idea: **compare each category with itself**, not with the others. A category with 1M clicks
and one with 10k clicks can be equally seasonal.

### 4.1 Seasonality index (per month)
$$\text{index}_{m} = \frac{\text{clicks}_{m}}{\text{mean(clicks)}}$$

1.0 = an average month, 1.5 = 50 % above average, 0.7 = 30 % below.

In [ ]:
seasonal = monthly.div(monthly.mean(axis=1), axis=0)
seasonal.round(2).head()

### 4.2 Category KPIs

| KPI | Formula | Why |
|---|---|---|
| **Seasonality score** | `max(index) − min(index)` | Size of the yearly swing; the main ranking KPI. 0 = perfectly flat |
| **Peak / low month** | `argmax(index)` / `argmin(index)` | *When* to publish content |
| **Avg. monthly / total clicks** | `mean` / `sum` of clicks | *How big* the category is |
| **Peak ÷ low** | `max(clicks) ÷ min(clicks)` | Intuitive "how many times bigger" number for business users |
| **Filled values** | count of missing months | Transparency: how many months were estimated |

> **Iteration note:** the first exploration used the *standard deviation* of the index as the score.
> The app switched to **max − min**, which is easier to explain ("the best month is X above the worst").

In [ ]:
kpis = pd.DataFrame({
    "seasonality_score": seasonal.max(axis=1) - seasonal.min(axis=1),
    "peak_month": seasonal.idxmax(axis=1),
    "low_month": seasonal.idxmin(axis=1),
    "avg_monthly_clicks": monthly.mean(axis=1),
    "total_clicks": monthly.sum(axis=1),
    "peak_vs_low": monthly.max(axis=1) / monthly.min(axis=1),
    "filled_values": monthly_raw.isna().sum(axis=1),
}).round(2)

kpis.sort_values("seasonality_score", ascending=False).head()

### 4.3 Strength buckets and peak season
Numbers are bucketed into words business users understand:

- **flat** < 0.5 · **moderate** 0.5 – 1.0 · **strong** > 1.0

The peak month is mapped to its **meteorological season**. Flat categories get
**"no clear season"** — their "peak" is just noise.

In [ ]:
SEASON_MAP = {12: "winter", 1: "winter", 2: "winter",
              3: "spring", 4: "spring", 5: "spring",
              6: "summer", 7: "summer", 8: "summer",
              9: "autumn", 10: "autumn", 11: "autumn"}

kpis["strength"] = pd.cut(kpis["seasonality_score"], bins=[0, 0.5, 1.0, np.inf],
                          labels=["flat", "moderate", "strong"]).astype(str)
kpis["peak_season"] = kpis["peak_month"].map(SEASON_MAP)
kpis.loc[kpis["strength"] == "flat", "peak_season"] = "no clear season"

display(kpis["strength"].value_counts().to_frame("categories"))
display(kpis["peak_season"].value_counts().to_frame("categories"))

## 5. Priority groups & the long table for charts

### 5.1 Priority matrix (size × seasonality)
To turn the KPIs into an **action list**, categories are split at the **median** size and the
**median** score into four groups. *Big & seasonal* categories get campaign planning first.

In [ ]:
is_big = kpis["avg_monthly_clicks"] >= kpis["avg_monthly_clicks"].median()
is_seasonal = kpis["seasonality_score"] >= kpis["seasonality_score"].median()
kpis["priority"] = np.select(
    [is_big & is_seasonal, ~is_big & is_seasonal, is_big & ~is_seasonal],
    ["1. big & seasonal", "2. small & seasonal", "3. big & steady"],
    default="4. small & steady",
)
kpis["priority"].value_counts().sort_index().to_frame("categories")

### 5.2 Long ("tidy") table
Plotly Express works best with **one row per observation**. The wide tables are stacked back into a
long table (category × month) and enriched with the KPI labels, so any chart can colour or filter by
season, strength or priority.

In [ ]:
MONTH_NAMES = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
               7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}

monthly_long = (monthly.stack().rename("clicks").to_frame()
                .join(seasonal.stack().rename("index"))
                .reset_index())
monthly_long["month_name"] = monthly_long["month"].map(MONTH_NAMES)
monthly_long = monthly_long.merge(kpis[["peak_season", "strength", "priority"]],
                                  left_on="category", right_index=True)
print(monthly_long.shape)
monthly_long.head()

### 5.3 Packaging it for the app
All of sections 3 – 5 live in a single function in `utils.py`. The `@st.cache_data` decorator makes
Streamlit compute it **once** and reuse the result on every click, instead of re-reading the CSV
each time a widget changes.

```python
@st.cache_data  # calculate once, not on every click
def load_data():
    df = pd.read_csv("data/combined_categories.csv")
    ...
    return monthly, seasonal, kpis, monthly_long
```

The app is started from the project folder, so the relative path `data/...` resolves correctly.

## 6. Visual design system

### 6.1 Color rules
- **Color follows the entity, never its rank:** each season always has the same color on every page.
- **Sequential blue ramp** for magnitudes (clicks).
- **Diverging blue ↔ gray ↔ red** for the seasonality index, centred on **1 = average month**.

In [ ]:
SEASON_ORDER = ["winter", "spring", "summer", "autumn", "no clear season"]
SEASON_COLORS = {"winter": "#2a78d6", "summer": "#eb6834", "spring": "#1baf7a",
                 "autumn": "#eda100", "no clear season": "#898781"}
BLUE_SCALE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
DIVERGING_SCALE = [[0.0, "#184f95"], [0.25, "#6da7ec"], [0.5, "#f0efec"],
                   [0.75, "#ec835a"], [1.0, "#b8312f"]]
MUTED, GRID, AXIS = "#898781", "#e1e0d9", "#c3c2b7"

### 6.2 A custom Plotly template
Instead of styling every chart, one template named `smartbuy` is registered and set as default.
Every chart in the app automatically gets the same font, transparent background, light grid and
horizontal legend.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

tpl = go.layout.Template()
tpl.layout = go.Layout(
    font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif', color="#52514e", size=13),
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    margin=dict(l=10, r=10, t=30, b=10),
    xaxis=dict(gridcolor=GRID, linecolor=AXIS, zeroline=False, tickfont=dict(color=MUTED)),
    yaxis=dict(gridcolor=GRID, linecolor=AXIS, zeroline=False, tickfont=dict(color=MUTED)),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, title_text=""),
    colorway=list(SEASON_COLORS.values()),
    bargap=0.25,
)
pio.templates["smartbuy"] = tpl
pio.templates.default = "plotly_white+smartbuy"   # in the app: "smartbuy"

### 6.3 Theme and custom CSS
- `.streamlit/config.toml` sets the base theme: light background `#f5f7fb`, primary indigo `#6366f1`,
  minimal toolbar.
- `ui.py` injects one CSS block (`setup_page()`) for the gradient **hero banner**, white **cards**
  with rounded corners, styled **KPI tiles** and the sidebar.
- Small helpers render these components so the pages stay short: `hero()`, `kpi_cards()`,
  `section()`, `chart_card()` and `style_fig()` (one consistent look for every Plotly chart).

```toml
[theme]
base = "light"
primaryColor = "#6366f1"
backgroundColor = "#f5f7fb"
secondaryBackgroundColor = "#eef0f8"
textColor = "#1e293b"
```

## 7. App architecture: `app.py`

`app.py` is deliberately short. It does four things:

### 7.1 Page configuration and styling
```python
setup_page("smartbuy Seasonality", "📅")   # st.set_page_config(layout="wide") + custom CSS
```

### 7.2 Load the data once
```python
monthly, seasonal, kpis, monthly_long = load_data()   # cached with @st.cache_data
```

### 7.3 Multipage navigation
The modern `st.navigation` / `st.Page` API defines the pages explicitly (titles, icons, default page)
instead of relying on file names in a `pages/` folder:

```python
home = st.Page("views/home.py", title="Project summary", icon="🏠", default=True)
pages = [
    home,
    st.Page("views/kpis.py", title="KPIs", icon="📊"),
    st.Page("views/top_clicks.py", title="Top clicks", icon="🏆"),
    st.Page("views/seasonality.py", title="It's all about seasonality", icon="🌦️"),
]
page = st.navigation(pages)
```

### 7.4 Shared sidebar filters, then run the page
```python
render_sidebar(kpis, show_filters=page is not home)
page.run()
```
The sidebar (in `ui.py`) offers a **category search**, **peak season** and **seasonality strength**
pills, a **priority group** selector, a **peak month** range and an **average clicks** range, plus a
counter ("42 of 100 categories shown") and a reset button. Every widget stores its value in
`st.session_state`, so **the same filters apply to every data page**.

### 7.5 How the pages read the filters
Each data page calls `apply_filters(kpis)` from `ui.py`, which reads the sidebar values from
`session_state` and returns only the matching categories. Below is a plain-pandas version of the
same logic (search, season and strength):

In [ ]:
def apply_filters_demo(kpis, search="", seasons=(), strengths=()):
    # In the app these values come from st.session_state (f_search, f_season, f_strength)
    f = kpis
    if search.strip():
        f = f[f.index.str.contains(search.strip(), case=False, regex=False)]
    if seasons:
        f = f[f["peak_season"].isin(seasons)]
    if strengths:
        f = f[f["strength"].isin(strengths)]
    return f

# Example: strongly seasonal summer categories
apply_filters_demo(kpis, seasons=["summer"], strengths=["strong"])[["seasonality_score", "peak_month"]]

,seasonality_score,peak_month
category,,
brushcutters,2.29,6
cool boxes,1.98,7
garden hoses,2.04,7
men's shirts,1.81,7
sun care,2.25,6
trekking sandals,2.23,6
umbrellas,2.47,6


If the filters match no category, the page shows a friendly message via `empty_state()` and stops,
so no chart crashes on an empty selection.

## 8. The four pages

Each subsection below describes one page and reproduces its main chart.

### 8.1 🏠 Project summary (`views/home.py`)
- **Hero banner** with the question the app answers.
- **Four headline tiles**: categories, months, total clicks, strongly seasonal categories.
- **The challenge / the solution** cards and a "How we did it" section.
- **Navigation cards** with `st.page_link` to the other pages.

In [ ]:
month_share = monthly.sum() / monthly.sum().sum() * 100
peak_counts = kpis["peak_month"].value_counts()

print("Categories analysed:     ", len(kpis))
print("Show a seasonal pattern: ", f"{(kpis['strength'] != 'flat').mean() * 100:.0f}%")
print("Busiest month:           ", MONTH_NAMES[month_share.idxmax()],
      f"({month_share.max():.1f}% of yearly clicks)")
print("Most common peak month:  ", MONTH_NAMES[peak_counts.idxmax()],
      f"({peak_counts.max()} categories)")

### 8.2 📊 KPIs (`views/kpis.py`)
- KPI tiles for the current filter selection.
- **When is the shop busiest?** Share of clicks per month, with a line at the average month.
- **Which categories matter most?** Priority matrix: seasonality score vs. average clicks, with median lines.
- **Priority list**: a sortable table of the chosen priority groups.

In [ ]:
fig = px.scatter(kpis.reset_index(), x="seasonality_score", y="avg_monthly_clicks",
                 color="peak_season", color_discrete_map=SEASON_COLORS,
                 category_orders={"peak_season": SEASON_ORDER},
                 hover_name="category", hover_data={"priority": True, "peak_season": False},
                 log_y=True,
                 labels={"seasonality_score": "Seasonality score",
                         "avg_monthly_clicks": "Avg. monthly clicks (log)"},
                 title="Priority matrix — top right = big & seasonal")
fig.update_traces(marker=dict(size=10, line=dict(color="white", width=1.5), opacity=.9))
fig.add_vline(x=kpis["seasonality_score"].median(), line_dash="dash", line_color=MUTED)
fig.add_hline(y=kpis["avg_monthly_clicks"].median(), line_dash="dash", line_color=MUTED)
fig.update_layout(height=420)
fig.show()

### 8.3 🏆 Top clicks (`views/top_clicks.py`)
- A **slider** chooses n (its maximum adapts to the filtered selection).
- Tiles: clicks of the top n, their share of all traffic, the #1 category.
- **Horizontal bar chart** coloured by peak season, plus an expandable table.

Here with n = 10:

In [ ]:
n = 10
top = kpis.nlargest(n, "total_clicks").reset_index()
top["peak_month_name"] = top["peak_month"].map(MONTH_NAMES)

fig = px.bar(top, x="total_clicks", y="category", orientation="h", color="peak_season",
             color_discrete_map=SEASON_COLORS, category_orders={"peak_season": SEASON_ORDER},
             text=top["total_clicks"].map(lambda v: f"{v / 1e6:.1f}M"),
             labels={"total_clicks": "Total clicks 2021", "category": ""},
             title=f"Top {n} categories by clicks")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_yaxes(categoryorder="total ascending")
fig.update_xaxes(showgrid=False, showticklabels=False, title=None)
fig.update_layout(height=34 * n + 120)
fig.show()

print(f"Top {n} share of all clicks: {top['total_clicks'].sum() / kpis['total_clicks'].sum() * 100:.0f}%")

### 8.4 🌦️ It's all about seasonality (`views/seasonality.py`)
Organised in **four tabs** to avoid an endless scroll:

1. **🌦️ Season patterns**: the average yearly shape of each season group.
2. **🔥 Most seasonal**: the top seasonal categories (slider).
3. **🔍 Category deep dive**: pick one category and see its monthly clicks or seasonality index.
4. **🔗 Compare & similar**: the app pre-selects the categories with the **most similar monthly
   pattern** (highest correlation) for comparison.

The heatmap below gives an overview of all 100 seasonality profiles at once (notebook view):

In [ ]:
order = kpis.sort_values(["peak_month", "seasonality_score"], ascending=[True, False]).index
heat = seasonal.loc[order].rename(columns=MONTH_NAMES)
span = max(abs(heat.values.max() - 1), abs(1 - heat.values.min()))   # symmetric around 1

fig = px.imshow(heat, aspect="auto", color_continuous_scale=DIVERGING_SCALE,
                zmin=1 - span, zmax=1 + span, labels={"x": "", "y": "", "color": "Index"},
                title="Seasonality index (red = above average, blue = below)")
fig.update_traces(xgap=1, ygap=1)
fig.update_layout(height=16 * len(heat) + 120)
fig.update_yaxes(tickfont=dict(size=9))
fig.show()

**Deep dive — finding similar categories.** The correlation between the 12-month index profiles
tells which categories move together. This drives the default comparison in tab 4:

In [ ]:
category = "sun care" if "sun care" in seasonal.index else seasonal.index[0]
similar = seasonal.T.corr()[category].drop(category).nlargest(3)
print(f"Most similar to '{category}':")
similar.round(3)

In [ ]:
MONTH_ORDER = list(MONTH_NAMES.values())
comp = seasonal.loc[[category] + similar.index.tolist()].rename(columns=MONTH_NAMES)
comp = comp.reset_index().melt(id_vars="category", var_name="month_name", value_name="index")

fig = px.line(comp, x="month_name", y="index", color="category", markers=True,
              category_orders={"month_name": MONTH_ORDER},
              labels={"month_name": "", "index": "Seasonality index", "category": ""},
              title=f"'{category}' vs. its most similar categories")
fig.update_traces(selector=dict(name=category), line_width=4)
fig.add_hline(y=1, line_dash="dash", line_color=MUTED)
fig.update_layout(height=420, hovermode="x unified")
fig.show()

## 9. Performance & robustness

| Concern | Solution in the app |
|---|---|
| Re-reading/processing data on every click | `@st.cache_data` on `load_data()` |
| 100 files to read at start-up | Pre-combined into `combined_categories.csv` |
| Missing months break the 12-month profile | Circular interpolation (`fill_circular`) |
| Filters match no category | `empty_state()` shows a message and stops the page |
| Slider with only 1 category | Slider is replaced by a fixed value |
| Same KPI computed differently on two pages | All logic centralised in `utils.py` |
| Inconsistent chart styling | One `style_fig()` helper and fixed season colors in `ui.py` |
| Flat categories with a random "peak" | Labelled *no clear season* |

## 10. How to run it & lessons learned

### 10.1 Run the app
```bash
cd smartbuy-seasonality-dashboard
pip install -r requirements.txt
streamlit run app.py
```
The app opens at `http://localhost:8501`.

### 10.2 Development process
1. **Explore** the data in a notebook (`01_exploration.ipynb`): structure, missing values,
   first charts.
2. **Define KPIs** and test them (`02_seasonality_analysis.ipynb`).
3. **First app version** (now in `app_v1/`): an overview, a category explorer and a seasonal groups page.
4. **Refactor** into `app.py` + `ui.py` + `utils.py` + `views/`, with the `st.navigation` API.
5. **Add shared sidebar filters**, a consistent design system and explanatory cards.

### 10.3 Lessons learned
- **Normalise first.** Comparing each category with its own average makes seasonality comparable
  across sizes.
- **Choose explainable KPIs.** *max − min* beat the standard deviation because it is easy to say out loud.
- **Translate numbers into words.** Buckets (flat / moderate / strong) and priority groups make the
  results actionable for the Content Department.
- **One source of truth.** Centralising the logic in `utils.py` kept all pages consistent.
- **Consistent colors.** A fixed color per season lets users read any chart without checking the legend.